In [54]:
# ============================================================
# FASE 1: LIMPIEZA DE DATOS (OPTIMIZADA PARA KAGGLE)
# ============================================================

library(readxl)
library(dplyr)
install.packages("moments")
library(moments)

# ── 1. CARGA DE DATOS ────────────────────────────────────────
train <- read_excel("Datos Taller individual - 2410 (Estudiantes).xlsx", sheet = "Train")
test  <- read_excel("Datos Taller individual - 2410 (Estudiantes).xlsx", sheet = "Test")

cat("Train:", dim(train), "\n")
cat("Test :", dim(test), "\n")

# ── 2. VARIABLES ─────────────────────────────────────────────
num_vars <- c("correo","paginas","telefono","impresa",
              "servicio","edadloc","nomina")

cat_vars <- c("idmercado","promo","tamamer")

# ── 3. IMPUTACIÓN (rápida y vectorizada) ─────────────────────
train[num_vars] <- lapply(train[num_vars], function(x){
  x[is.na(x)] <- median(x, na.rm = TRUE)
  x
})

test[num_vars] <- lapply(test[num_vars], function(x){
  x[is.na(x)] <- median(x, na.rm = TRUE)
  x
})

mode_fast <- function(x){
  ux <- unique(x)
  ux[which.max(tabulate(match(x, ux)))]
}

train[cat_vars] <- lapply(train[cat_vars], function(x){
  x[is.na(x)] <- mode_fast(x)
  x
})

test[cat_vars] <- lapply(test[cat_vars], function(x){
  x[is.na(x)] <- mode_fast(x)
  x
})

# ── 4. FACTORES ──────────────────────────────────────────────
train[cat_vars] <- lapply(train[cat_vars], as.factor)
test[cat_vars]  <- lapply(test[cat_vars], as.factor)

# 🔥 alineación de niveles (CRÍTICO)
for (v in cat_vars) {
  test[[v]] <- factor(test[[v]], levels = levels(train[[v]]))
}

# ── 5. WINSORIZACIÓN (vectorizada, rápida) ───────────────────
winsorize <- function(x){
  q <- quantile(x, c(0.01, 0.99), na.rm = TRUE)
  pmax(pmin(x, q[2]), q[1])
}

train[num_vars] <- lapply(train[num_vars], winsorize)
test[num_vars]  <- lapply(test[num_vars], winsorize)

# ── 6. CHECK MÍNIMO (sin gráficos pesados) ───────────────────
sk <- skewness(train$ropamujer)

cat("\nSkewness ropamujer:", round(sk, 3), "\n")

cat("Distribución:",
    ifelse(abs(sk) < 0.5, "Simétrica",
    ifelse(abs(sk) < 1, "Moderada", "Alta asimetría")),
    "\n")

# ── 7. RESUMEN FINAL ─────────────────────────────────────────
cat("\n═══════════════════════\n")
cat("DATOS LISTOS ✓\n")
cat("Train:", nrow(train), "x", ncol(train), "\n")
cat("Test :", nrow(test), "x", ncol(test), "\n")
cat("NAs Train:", sum(is.na(train)), "\n")
cat("═══════════════════════\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Train: 1600 12 
Test : 400 12 

Skewness ropamujer: 0.312 
Distribución: Simétrica 

═══════════════════════
DATOS LISTOS ✓
Train: 1600 x 12 
Test : 400 x 12 
NAs Train: 0 
═══════════════════════


In [ ]:
library(dplyr)
install.packages("MASS")
library(MASS)
install.packages("caret")
library(caret)
install.packages("glmnet")
library(glmnet)

select <- dplyr::select
filter <- dplyr::filter

set.seed(42)
ctrl <- trainControl(method = "cv", number = 10)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
# ============================================================
# RONDA 4: AFINAMIENTO FINO DEL MODELO GANADOR
# ============================================================

# ── 1. VER CUÁLES VARIABLES FALTAN POR EXPLOTAR ──────────────
# idmercado NO entró al stepwise — puede aportar como interacción
# nomina tampoco — pero su correlación es 0, dejémosla fuera
# correo no está directo — solo en interacciones

# ── 2. FEATURE ENGINEERING v4: REFINAMIENTO ──────────────────
feature_engineering_v4 <- function(df, ref = train) {

  pct <- function(var, p) quantile(ref[[var]], p, na.rm = TRUE)

  df %>% mutate(

    # === SPLINES DE SERVICIO (ya funcionaron bien) ===
    srv_k1 = pmax(0, servicio - pct("servicio", 0.20)),
    srv_k2 = pmax(0, servicio - pct("servicio", 0.40)),
    srv_k3 = pmax(0, servicio - pct("servicio", 0.60)),
    srv_k4 = pmax(0, servicio - pct("servicio", 0.80)),

    # === SPLINES CORREO ===
    cor_k1 = pmax(0, correo - pct("correo", 0.20)),
    cor_k2 = pmax(0, correo - pct("correo", 0.40)),
    cor_k3 = pmax(0, correo - pct("correo", 0.60)),
    cor_k4 = pmax(0, correo - pct("correo", 0.80)),

    # === SPLINES PÁGINAS ===
    pag_k1 = pmax(0, paginas - pct("paginas", 0.20)),
    pag_k2 = pmax(0, paginas - pct("paginas", 0.40)),
    pag_k3 = pmax(0, paginas - pct("paginas", 0.60)),
    pag_k4 = pmax(0, paginas - pct("paginas", 0.80)),

    # === SPLINES IMPRESA ===
    imp_k1 = pmax(0, impresa - pct("impresa", 0.20)),
    imp_k2 = pmax(0, impresa - pct("impresa", 0.40)),
    imp_k3 = pmax(0, impresa - pct("impresa", 0.60)),
    imp_k4 = pmax(0, impresa - pct("impresa", 0.80)),

    # === SPLINES TELEFONO ===
    tel_k1 = pmax(0, telefono - pct("telefono", 0.20)),
    tel_k2 = pmax(0, telefono - pct("telefono", 0.40)),
    tel_k3 = pmax(0, telefono - pct("telefono", 0.60)),
    tel_k4 = pmax(0, telefono - pct("telefono", 0.80)),

    # === SPLINES EDADLOC ===
    edad_k1 = pmax(0, edadloc - pct("edadloc", 0.20)),
    edad_k2 = pmax(0, edadloc - pct("edadloc", 0.40)),
    edad_k3 = pmax(0, edadloc - pct("edadloc", 0.60)),
    edad_k4 = pmax(0, edadloc - pct("edadloc", 0.80)),

    # === INTERACCIONES GANADORAS (las que entró el stepwise) ===
    srv_x_cor  = servicio * correo,
    srv_x_tel  = servicio * telefono,
    srv_x_imp  = servicio * impresa,
    srv_x_pag  = servicio * paginas,
    srv_x_edad = servicio * edadloc,
    cor_x_tel  = correo * telefono,
    cor_x_pag  = correo * paginas,
    cor_x_imp  = correo * impresa,
    srv_cor_tel = servicio * correo * telefono,
    srv_cor_pag = servicio * correo * paginas,

    # === CUADRÁTICOS GANADORES ===
    srv2  = servicio^2,
    cor2  = correo^2,
    pag2  = paginas^2,
    tel2  = telefono^2,
    imp2  = impresa^2,
    edad2 = edadloc^2,

    # === CÚBICOS (nuevo) ===
    srv3  = servicio^3,
    cor3  = correo^3,
    pag3  = paginas^3,

    # === LOGS ===
    log_srv  = log1p(servicio),
    log_cor  = log1p(correo),
    log_imp  = log1p(impresa),
    log_edad = log1p(edadloc),
    log_tel  = log1p(telefono),

    # === RATIOS ===
    srv_per_correo = servicio / (correo / 1000),
    imp_per_correo = impresa  / (correo / 1000),
    pag_per_tel    = paginas  / (telefono + 1),
    srv_per_pag    = servicio / (paginas + 1),

    # === ENCODING ORDINAL ===
    tamamer_ord = case_when(
      tamamer == "Pequeño" ~ 1,
      tamamer == "Median"  ~ 2,
      tamamer == "Grande"  ~ 3
    ),
    tam_x_srv  = tamamer_ord * servicio,
    tam_x_cor  = tamamer_ord * correo,
    tam_x_pag  = tamamer_ord * paginas,
    tam_x_imp  = tamamer_ord * impresa,

    # === MERCADO (nuevo: interacciones más ricas) ===
    idmercado_num = as.numeric(as.character(idmercado)),
    mkt_x_srv     = idmercado_num * servicio,
    mkt_x_cor     = idmercado_num * correo,
    mkt_x_pag     = idmercado_num * paginas,
    mkt_x_srv2    = idmercado_num * servicio^2,

    # === PROMO ===
    promo_num   = as.numeric(as.character(promo)),
    promo_x_srv = promo_num * servicio,
    promo_x_cor = promo_num * correo,
    promo_x_pag = promo_num * paginas,
    promo_x_imp = promo_num * impresa,

    # === NUEVAS: INTERACCIONES SPLINE × FACTOR ===
    # ¿El efecto de servicio varía por tipo de mercado?
    mkt_x_srv_k4  = idmercado_num * pmax(0, servicio - pct("servicio", 0.80)),
    tam_x_srv_k4  = tamamer_ord   * pmax(0, servicio - pct("servicio", 0.80)),
    promo_x_srv_k4 = promo_num    * pmax(0, servicio - pct("servicio", 0.80)),

    # === NUEVA: ÍNDICE COMPUESTO DE MARKETING ===
    # Combina todos los esfuerzos de marketing en un índice
    marketing_idx = scale(correo)[,1] + scale(paginas)[,1] +
                    scale(telefono)[,1] + scale(impresa)[,1] +
                    scale(servicio)[,1],
    mkt_idx2 = marketing_idx^2
  )
}

train_fe4 <- feature_engineering_v4(train, ref = train)
test_fe4  <- feature_engineering_v4(test,  ref = train)

# Winsorizar
vars_wins4 <- setdiff(names(train_fe4),
                      c("idloc","ropamujer","idmercado","promo","tamamer"))
train_fe4 <- train_fe4 %>% mutate(across(all_of(vars_wins4), winsorize))
test_fe4  <- test_fe4  %>% mutate(across(all_of(vars_wins4), winsorize))

cat("Variables v4:", ncol(train_fe4), "\n")

# ── Preparar ─────────────────────────────────────────────────
y_train   <- train_fe4$ropamujer
df_model4 <- dplyr::select(train_fe4, -idloc, -ropamujer) %>%
             mutate(ropamujer = y_train)
X_test4   <- dplyr::select(test_fe4, -idloc, -ropamujer)

# ── Stepwise v4 ──────────────────────────────────────────────
cat("\n── Stepwise OLS v4 ──\n")
lm_full4 <- lm(ropamujer ~ ., data = df_model4)
lm_step4 <- stepAIC(lm_full4, direction = "both", trace = FALSE)

r2_step4   <- summary(lm_step4)$adj.r.squared
pred_s4    <- predict(lm_step4, df_model4)
rmse_s4    <- sqrt(mean((y_train - pred_s4)^2))

cat("R² ajustado:", round(r2_step4, 4), "\n")
cat("RMSE Train: ", round(rmse_s4, 2), "\n")
cat("Variables:  ", length(coef(lm_step4)) - 1, "\n")

set.seed(42)
cv_step4 <- train(formula(lm_step4), data = df_model4,
                  method = "lm", trControl = ctrl)
cat("RMSE CV 10-fold:", round(cv_step4$results$RMSE, 2), "\n")

# ── LASSO v4 ─────────────────────────────────────────────────
X_mat4_train <- model.matrix(ropamujer ~ ., data = df_model4)[, -1]
X_mat4_test  <- model.matrix(~ ., data = X_test4)[, -1]

missing4 <- setdiff(colnames(X_mat4_train), colnames(X_mat4_test))
for (col in missing4) X_mat4_test <- cbind(X_mat4_test, setNames(data.frame(0), col))
X_mat4_test <- X_mat4_test[, colnames(X_mat4_train)]

set.seed(42)
cv_lasso4 <- cv.glmnet(X_mat4_train, y_train, alpha = 1, nfolds = 10)
pred_l4   <- predict(cv_lasso4, X_mat4_train, s = "lambda.min")
rmse_l4   <- sqrt(mean((y_train - pred_l4)^2))
rmse_cv_l4 <- sqrt(min(cv_lasso4$cvm))
cat("\nRMSE Train LASSO v4:", round(rmse_l4, 2), "\n")
cat("RMSE CV   LASSO v4:", round(rmse_cv_l4, 2), "\n")

# ── Resumen progreso ──────────────────────────────────────────
cat("\n══════════════════════════════════════════════\n")
cat("  PROGRESO COMPLETO\n")
cat("══════════════════════════════════════════════\n")
cat(sprintf("Ronda 1 - Stepwise CV:   %8.2f\n", 10776.79))
cat(sprintf("Ronda 2 - Stepwise CV:   %8.2f\n", 10655.56))
cat(sprintf("Ronda 3 - Stepwise CV:   %8.2f\n", 10460.63))
cat(sprintf("Ronda 4 - Stepwise CV:   %8.2f\n", cv_step4$results$RMSE))
cat(sprintf("Ronda 4 - LASSO CV:      %8.2f\n", rmse_cv_l4))
cat("══════════════════════════════════════════════\n")

# ── Exportar ganador ──────────────────────────────────────────
use_step4 <- cv_step4$results$RMSE <= rmse_cv_l4

if (use_step4) {
  pred_kaggle4 <- predict(lm_step4, newdata = X_test4)
  cat("Modelo elegido: Stepwise OLS v4\n")
} else {
  pred_kaggle4 <- predict(cv_lasso4, X_mat4_test, s = "lambda.min") %>% as.vector()
  cat("Modelo elegido: LASSO v4\n")
}

submission_v4 <- data.frame(
  idloc     = test$idloc,
  ropamujer = pred_kaggle4
)

write.csv(submission_v4, "predicciones_v4.csv", row.names = FALSE)
cat("predicciones_v4.csv generado ✓\n")
print(head(submission_v4))

In [ ]:
install.packages("car")
library(car)
# ============================================================
# RONDA 7: ATACAR LOS PATRONES ENCONTRADOS
# ============================================================

# ── 1. INVESTIGAR PROMO=2 ─────────────────────────────────────
cat("── Distribución ropamujer por promo ──\n")
train %>%
  group_by(promo) %>%
  summarise(
    n      = n(),
    media  = round(mean(ropamujer), 0),
    mediana= round(median(ropamujer), 0),
    sd     = round(sd(ropamujer), 0),
    min    = round(min(ropamujer), 0),
    max    = round(max(ropamujer), 0),
    cv_pct = round(sd(ropamujer)/mean(ropamujer)*100, 1)
  ) %>% print()

cat("\n── ¿Promo=2 tiene mayor varianza? ──\n")
leveneTest(ropamujer ~ promo, data = train)

cat("\n── Distribución de ropamujer por promo (resumen) ──\n")
tapply(train$ropamujer, train$promo, quantile, probs=c(0.05,0.25,0.5,0.75,0.95))

# ── 2. INVESTIGAR CORREO BAJO ────────────────────────────────
cat("\n── Tiendas con correo < 3000 ──\n")
train %>% filter(correo < 3000) %>%
  dplyr::select(idloc, correo, ropamujer, promo, idmercado, tamamer) %>%
  print()

cat("\n── ¿Cuántas en test? ──\n")
test %>% filter(correo < 3000) %>% nrow()

# ── 3. MODELO v5 ─────────────────────────────────────────────
cat("\n══ MODELO v5: Promo como modificador ══\n")

train_fe5 <- train_fe4 %>%
  mutate(
    es_promo2     = as.integer(promo == 2),
    es_promo2_srv = es_promo2 * servicio,
    es_promo2_cor = es_promo2 * correo,
    es_promo2_pag = es_promo2 * paginas,
    es_promo2_imp = es_promo2 * impresa,
    es_promo2_tel = es_promo2 * telefono,
    es_promo2_srv2 = es_promo2 * servicio^2,
    es_promo2_cor2 = es_promo2 * correo^2,
    es_promo2_srv_k2 = es_promo2 * pmax(0, servicio - quantile(train$servicio, 0.40)),
    es_promo2_srv_k3 = es_promo2 * pmax(0, servicio - quantile(train$servicio, 0.60)),
    es_promo2_srv_k4 = es_promo2 * pmax(0, servicio - quantile(train$servicio, 0.80)),
    correo_bajo      = as.integer(correo < 3000),
    correo_bajo_srv  = correo_bajo * servicio
  )

test_fe5 <- test_fe4 %>%
  mutate(
    es_promo2     = as.integer(promo == 2),
    es_promo2_srv = es_promo2 * servicio,
    es_promo2_cor = es_promo2 * correo,
    es_promo2_pag = es_promo2 * paginas,
    es_promo2_imp = es_promo2 * impresa,
    es_promo2_tel = es_promo2 * telefono,
    es_promo2_srv2 = es_promo2 * servicio^2,
    es_promo2_cor2 = es_promo2 * correo^2,
    es_promo2_srv_k2 = es_promo2 * pmax(0, servicio - quantile(train$servicio, 0.40)),
    es_promo2_srv_k3 = es_promo2 * pmax(0, servicio - quantile(train$servicio, 0.60)),
    es_promo2_srv_k4 = es_promo2 * pmax(0, servicio - quantile(train$servicio, 0.80)),
    correo_bajo      = as.integer(correo < 3000),
    correo_bajo_srv  = correo_bajo * servicio
  )

# 🔥 WINSORIZACIÓN (AJUSTE CLAVE)
# 🔥 WINSORIZACIÓN MEJORADA (2% – 98%)
lim_inf <- quantile(train_fe5$ropamujer, 0.02)
lim_sup <- quantile(train_fe5$ropamujer, 0.98)

train_fe5$ropamujer <- pmax(train_fe5$ropamujer, lim_inf)
train_fe5$ropamujer <- pmin(train_fe5$ropamujer, lim_sup)

y_train   <- train_fe5$ropamujer
df_model5 <- dplyr::select(train_fe5, -idloc, -ropamujer) %>%
             mutate(ropamujer = y_train)
X_test5   <- dplyr::select(test_fe5, -idloc, -ropamujer)

# Stepwise con BIC
lm_full5 <- lm(ropamujer ~ ., data = df_model5)
n <- nrow(df_model5)
lm_step5 <- stepAIC(lm_full5, direction = "both", trace = FALSE, k = log(n))

r2_s5   <- summary(lm_step5)$adj.r.squared
pred_s5 <- predict(lm_step5, df_model5)
rmse_s5 <- sqrt(mean((y_train - pred_s5)^2))

cat("R² ajustado:", round(r2_s5, 4), "\n")
cat("RMSE Train:", round(rmse_s5, 2), "\n")

set.seed(42)
cv_step5 <- train(formula(lm_step5), data = df_model5,
                  method = "lm", trControl = ctrl)
cat("RMSE CV:", round(cv_step5$results$RMSE, 2), "\n")

# ── 4. EXPORTACIÓN ───────────────────────────────────────────

pred_test <- predict(lm_step5, newdata = X_test5)

submission <- data.frame(
  idloc     = test$idloc,
  ropamujer = pred_test
)

write.csv(submission, "predicciones_v6_winsor.csv", row.names = FALSE)

cat("predicciones_v6_winsor.csv generado ✓\n")
print(head(submission))